### Установка `PostgreSQL`


#### `Ubuntu`

```bash
sudo apt update
sudo apt install postgresql
```
Сразу после установки `PostgreSQL` автоматически запускается от имени вновь созданного пользователя `postgres`. Пользователь `postgres` понадобится нам дальше. Также будет создана база данных с именем `postgres`.



##### `Проверка правильности установки`

Посмотрим из командной строки, запущен ли PostgreSQL:
```bash
ps aux | grep postgres
```
--- 
> `ps` — это утилита для просмотра текущих процессов. Флаги в `aux` означают:

>>  - `a` — показать процессы всех пользователей (не только текущего);
>> - `u` — вывести подробный формат (пользователь, загрузка CPU/памяти, время запуска, команда и т. д.);
>>  - `x` — включить процессы, не привязанные к терминалу (например, службы и демоны).

> В результате `ps aux` выводит длинный список всех процессов в системе с подробной информацией по каждому.

> `grep` — утилита поиска текста. Здесь она фильтрует вывод `ps aux`, оставляя только строки, где встречается подстрока `postgres`. Так получают только процессы, относящиеся к `PostgreSQL`.

> Символ `|` передаёт вывод первой команды `(ps aux)` на вход второй `(grep postgres)`. То есть:

  >> `ps aux` формирует полный список процессов.
  >> Этот список сразу «скармливается» команде `grep`.
  >> `grep` оставляет только строки с `postgres` и показывает их в терминале.

---

Вывод может быть другим/ Главное, чтобы в нем встречалась строчка с текстом: 

---
```bash
bin/postgres
postgres  3437  0.0  2.3 295008 24160 ?        S    12:01   0:00 /usr/lib/postgresql/9.5/bin/postgres -D /var/lib/postgresql/9.5/main -c config_file=/etc/postgresql/9.5/main/postgresql.conf
```
---
В вашем случае вывод может отличаться, но главное — увидеть ту строчку, в которой отображается запуск самой базы данных. Если база данных запущена, то дальше нужно попробовать к ней подключиться. 
`PostgreSQL` поставляется со специальной программой `psql`, которая представляет собой интерактивную оболочку `(REPL)`. Через него можно управлять `PostgreSQL` и выполнять запросы к базам данных прямо из терминала:


##### `Подключение PostgreSQL`

Если `PostgreSQL` запустить без аргументов, то она пытается подключиться к локальной базе данных, которая находится на той же машине. При этом программа подключается с именем, которое совпадает с именем текущего пользователя. Делает она это с помощью роли с этим же именем. Если `PostgreSQL` установили верно, то запуск в `Linux`, например, `Ubuntu`, негативно реагирует на отсутствие соответствующей роли:

```bash
psql

psql: FATAL:  role "sergey" does not exist
```
Эту проблему можно решить с помощью конструкции `sudo -u postgres psql`, которую используют в терминале при подключении `PostgreSQL`. Но это не лучшее решение, по таким причинам:

- У пользователя `postgres` есть максимальные права в СУБД. Если завладеть ими, можно все уничтожить. Поэтому приложения или пользователи никогда не создают базы данных от имени `postgres` и никогда не работают из-под этого пользователя

- Придется постоянно использовать эту конструкцию `sudo -u postgres` для любых команд, которые связаны с СУБД. 

##### `Создание пользователя в СУБД`

Cоздадим роль с таким же именем, как и пользователь.

1. Посмотрите имя вашего текущего пользователя:

```bash
    whoami
```
    *sergey*
2. Создайте роль с таким же именем внутри `PostgreSQL` с помощью команды `createuser`. Ее нужно запускать от пользователя `postgres`, иначе она попробует соединиться с СУБД от имени текущего пользователя, которого там нет:

```bash
    # Флаг --createdb добавляет нашей роли возможность создавать базы данных
    # По умолчанию этой возможности нет
    sudo -u postgres createuser --createdb sergey

    # Чтобы удалить пользователя, можно воспользоваться командой:
    dropuser sergey
```
---

> Флаг `-u` в команде `sudo -u postgres` ... означает `«user»` — он задаёт пользователя, от имени которого будет выполнена команда.

> это значит: «запусти утилиту `createuser` от имени системного пользователя `postgres `(у него в `PostgreSQL` обычно есть права .
> суперпользователя), чтобы создать роль/пользователя `sergey` с правом создавать базы данных».

--- 

Теперь у нас есть роль в СУБД. Попробуем с ее помощью соединиться с `PostgreSQL`:

```bash
psql

psql: FATAL:  database "sergey" does not exist
```
Снова ошибка. Теперь `psql` «ругается», что не выбрана база данных. Невозможно соединиться с СУБД, если не указать конкретную базу данных. Ее можно указать самостоятельно — передать один аргумент в `psql`.

Мы уже знаем, что внутри PostgreSQL создана база postgres. Попробуем подключиться к ней:

```bash
psql postgres

postgres=>
```

Соединение удалось. Теперь посмотрим список ролей. Для этого подходит команда `\du (Describe Users)`, которую нужно выполнить внутри `REPL`:

```bash
postgres=> \du
                                   List of roles
 Role name |                         Attributes                         | Member of
-----------+------------------------------------------------------------+-----------
 postgres  | Superuser, Create role, Create DB, Replication, Bypass RLS | {}
 tirion    | Create DB                                                  | {}
```
В СУБД создалось две роли: `postgres` и которую самостоятельно добавили ранее. Теперь создадим базу данных.


##### `Создание базы данных в СУБД`

Для экспериментов нам понадобится база данных и, возможно, даже не одна. Создадим базу с именем hexlet. Сделать это можно из командной строки командой createdb:

```bash
# Опция --owner позволяет указать владельца создаваемой базы данных
sudo -u postgres createdb --owner=sergey hexlet
# Если запустить эту команду без аргументов,
# то она попытается создать базу данных,
# которая совпадает с именем вашего пользователя в системе
createdb hexlet
```
Имя для базы данных выбирается произвольно и обычно совпадает с названием проекта, для которого она создается. Имена баз уникальны в рамках одной СУБД. Это значит, что повторный вызов `createdb` с тем же именем приведет к ошибке.

После установки `PostgreSQL` создает несколько служебных баз данных, которые нужны для работы самой СУБД. Посмотрим их список:

```bash
# Посмотреть список баз данных
psql -l

# Неполный вывод
                          List of databases
   Name    |  Owner   | Encoding |   Collate   |    Ctype    |
-----------+----------+----------+-------------+-------------+
 pgadmin   | pgadmin  | UTF8     | en_US.UTF-8 | en_US.UTF-8 |
 tirion    | tirion   | UTF8     | en_US.UTF-8 | en_US.UTF-8 |
 postgres  | postgres | UTF8     | en_US.UTF-8 | en_US.UTF-8 |
 template0 | postgres | UTF8     | en_US.UTF-8 | en_US.UTF-8 |
 template1 | postgres | UTF8     | en_US.UTF-8 | en_US.UTF-8 |
```

Теперь у нас есть роль и база данных для экспериментов. Подключимся к этой базе данных:

```bash
psql hexlet

hexlet=>
```

Созданную базу данных можно удалить командой `dropdb`:

```bash
dropdb hexlet
```
Если удалили, не забудьте ее снова создать, так как она понадобится нам в дальнейшем.

Запускать команду `dropdb` нужно с осторожностью. Удаление базы данных — необратимый процесс, так как базы невозможно восстановить без резервных копий. Чтобы избежать проблем: команда `dropdb` не работает без аргументов — ей всегда нужно передавать имя базы.

Удалить базу данных можно только в том случае, если к ней никто не подключен — за исключением того, кто удаляет. Если есть другие клиенты, например `psql` или `pgadmin4`, то СУБД предупредит о невозможности выполнить команду.

```bash
dropdb hexlet
dropdb: error: database removal failed: ERROR:  database "hexlet" is being accessed by other users
DETAIL:  There is 1 other session using the database.
```

##### `Управляющие команды`

Внутри `psql` доступны разные управляющие команды. Все они построены по одному принципу — перед командой набирается обратный слеш `\`, например:

- Чтобы выйти, надо набрать `\q`
- Чтобы посмотреть полный список команд, надо набрать `\?`

Команда `\?` загружает пейджер, по которому можно перемещаться, используя стандартные комбинации текстового редактора Vim:

```bash
postgres=# \?
Informational
  (options: S = show system objects, + = additional detail)
  \d[S+]                 таблицы списков, представления и последовательности
  \d[S+]  NAME           описывать таблицу, представление, последовательность или индекс
  \da[S]  [PATTERN]      агрегаты списка
  \dA[+]  [PATTERN]      методы доступа к спискам
  \db[+]  [PATTERN]      список табличных областей
  \dc[S+] [PATTERN]      list conversions
  \dC[+]  [PATTERN]      list casts
  \dd[S]  [PATTERN]      show object descriptions not displayed elsewhere
  \dD[S+] [PATTERN]      list domains
  \ddp    [PATTERN]      list default privileges
  \dE[S+] [PATTERN]      list foreign tables
  \det[+] [PATTERN]      list foreign tables
  \des[+] [PATTERN]      list foreign servers
  \deu[+] [PATTERN]      list user mappings
  \dew[+] [PATTERN]      list foreign-data wrappers
  ...
```
Возможные проблемы

В работе с PostgreSQL вы можете столкнуться с такой проблемой:

```bash
psql

psql: could not connect to database template1: could not connect to server: No such file or directory
  Is the server running locally and accepting
  connections on Unix domain socket "/tmp/.s.PGSQL.5432"?
```

Этот вывод говорит, что подключение не удалось — нужно выяснять почему. В некоторых случаях такая ошибка возникает, если не запущен сервис `postgresql`. 
Тогда запустите его командой `sudo service postgresql start` и выполните подключение повторно.

Еще вам может встретиться ошибка `psql: command not found`. Она указывает, что вы ввели не существующую команду. Это может пройти по трем причинам:

  - В названии команды допущена опечатка
  - `PostgreSQL` установлен неверно
  - `PostgreSQL` не установлен вообще

